# Python Iterators

## Learning Objectives
- Understand the **iterator protocol** — `__iter__` and `__next__`
- Know the difference between an **iterable** (can produce an iterator) and an **iterator** (produces values one at a time)
- Build **custom iterators** using classes
- Recognise the built-in iterables: lists, dicts, files, `range`, etc.
- Use `itertools` to compose and transform iterators
- Know when to use a **custom iterator class** vs a **generator**

## Iterable vs Iterator

These two terms are related but distinct:

| Term | Must implement | Typical examples |
|---|---|---|
| **Iterable** | `__iter__()` → returns an iterator | `list`, `tuple`, `dict`, `str`, `set`, `range` |
| **Iterator** | `__iter__()` + `__next__()` | `enumerate`, `zip`, `map`, file objects, generators |

Key rule: **every iterator is an iterable** (its `__iter__` returns `self`), but **not every iterable is an iterator** (a `list` has no `__next__`).

### How Python's `for` loop works internally

```python
for x in obj:          # Python translates this to:
    use(x)

it = iter(obj)         # calls obj.__iter__()
while True:
    try:
        x = next(it)   # calls it.__next__()
        use(x)
    except StopIteration:
        break
```

In [12]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 340" width="740" height="340"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ar1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#555"/>
    </marker>
    <marker id="ar2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2166ac"/>
    </marker>
    <marker id="ar3" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
    <marker id="ar4" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#c0392b"/>
    </marker>
  </defs>
  <rect width="740" height="340" fill="#f9f9fb" rx="10"/>
  <text x="370" y="28" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Iterable vs Iterator</text>

  <!-- LEFT: Iterable -->
  <rect x="16" y="48" width="320" height="190" rx="10" fill="#fde8e8" stroke="#c0392b" stroke-width="2"/>
  <text x="176" y="74" text-anchor="middle" font-size="13" font-weight="bold" fill="#c0392b">Iterable</text>
  <text x="176" y="92" text-anchor="middle" font-size="10" fill="#c0392b">e.g. list, tuple, dict, str, range</text>
  <!-- list boxes -->
  <rect x="36"  y="104" width="38" height="34" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="55"  y="126" text-anchor="middle" font-size="13" fill="#7b0000">10</text>
  <rect x="80"  y="104" width="38" height="34" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="99"  y="126" text-anchor="middle" font-size="13" fill="#7b0000">20</text>
  <rect x="124" y="104" width="38" height="34" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="143" y="126" text-anchor="middle" font-size="13" fill="#7b0000">30</text>
  <rect x="168" y="104" width="38" height="34" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="187" y="126" text-anchor="middle" font-size="13" fill="#7b0000">40</text>
  <rect x="212" y="104" width="38" height="34" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="231" y="126" text-anchor="middle" font-size="13" fill="#7b0000">50</text>
  <!-- labels -->
  <rect x="36" y="150" width="275" height="22" rx="4" fill="#fde8e8"/>
  <text x="174" y="165" text-anchor="middle" font-size="10" fill="#c0392b">&#10003; implements  __iter__()  &#8594; returns an iterator</text>
  <rect x="36" y="177" width="275" height="22" rx="4" fill="#fde8e8"/>
  <text x="174" y="192" text-anchor="middle" font-size="10" fill="#c0392b">&#10007; does NOT implement  __next__()</text>
  <rect x="36" y="204" width="275" height="22" rx="4" fill="#fde8e8"/>
  <text x="174" y="219" text-anchor="middle" font-size="10" fill="#c0392b">can be iterated many times (creates new iterator each time)</text>

  <!-- Arrow: iter() -->
  <line x1="338" y1="143" x2="398" y2="143" stroke="#555" stroke-width="2" marker-end="url(#ar1)"/>
  <text x="368" y="136" text-anchor="middle" font-size="10" fill="#555">iter()</text>
  <text x="368" y="160" text-anchor="middle" font-size="9" fill="#888">__iter__()</text>

  <!-- RIGHT: Iterator -->
  <rect x="404" y="48" width="320" height="190" rx="10" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="2"/>
  <text x="564" y="74" text-anchor="middle" font-size="13" font-weight="bold" fill="#2e7d32">Iterator</text>
  <text x="564" y="92" text-anchor="middle" font-size="10" fill="#2e7d32">e.g. enumerate, zip, map, file, generator</text>
  <!-- cursor on boxes -->
  <rect x="424" y="104" width="38" height="34" rx="4" fill="#c8e6c9" stroke="#3a8c3f" stroke-width="1.2"/>
  <text x="443" y="126" text-anchor="middle" font-size="13" fill="#1b5e20">10</text>
  <rect x="468" y="104" width="38" height="34" rx="4" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.2" stroke-dasharray="4,2"/>
  <text x="487" y="126" text-anchor="middle" font-size="13" fill="#aaa">20</text>
  <rect x="512" y="104" width="38" height="34" rx="4" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.2" stroke-dasharray="4,2"/>
  <text x="531" y="126" text-anchor="middle" font-size="13" fill="#aaa">30</text>
  <rect x="556" y="104" width="38" height="34" rx="4" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.2" stroke-dasharray="4,2"/>
  <text x="575" y="126" text-anchor="middle" font-size="13" fill="#aaa">40</text>
  <rect x="600" y="104" width="38" height="34" rx="4" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.2" stroke-dasharray="4,2"/>
  <text x="619" y="126" text-anchor="middle" font-size="13" fill="#aaa">50</text>
  <!-- cursor arrow -->
  <polygon points="443,100 437,90 449,90" fill="#2e7d32"/>
  <text x="443" y="88" text-anchor="middle" font-size="8" fill="#2e7d32">pos</text>
  <!-- labels -->
  <rect x="424" y="150" width="275" height="22" rx="4" fill="#e8f5e9"/>
  <text x="562" y="165" text-anchor="middle" font-size="10" fill="#2e7d32">&#10003; implements  __iter__()  &#8594; returns self</text>
  <rect x="424" y="177" width="275" height="22" rx="4" fill="#e8f5e9"/>
  <text x="562" y="192" text-anchor="middle" font-size="10" fill="#2e7d32">&#10003; implements  __next__()  &#8594; next value / StopIteration</text>
  <rect x="424" y="204" width="275" height="22" rx="4" fill="#e8f5e9"/>
  <text x="562" y="219" text-anchor="middle" font-size="10" fill="#2e7d32">single-pass &#8212; cannot be reset without recreating</text>

  <!-- Venn at bottom -->
  <ellipse cx="370" cy="294" rx="200" ry="30" fill="#fde8e8" stroke="#c0392b" stroke-width="1.5"/>
  <text x="250" y="298" text-anchor="middle" font-size="10" fill="#c0392b">Iterables</text>
  <ellipse cx="400" cy="294" rx="120" ry="20" fill="#c8e6c9" stroke="#3a8c3f" stroke-width="1.5" opacity="0.85"/>
  <text x="420" y="298" text-anchor="middle" font-size="10" fill="#1b5e20">Iterators</text>
  <text x="370" y="330" text-anchor="middle" font-size="10" fill="#555">Every iterator is an iterable; not every iterable is an iterator.</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  Solid border = current position; dashed = not yet visited. The green triangle marks the iterator cursor.
</figcaption>
</figure>
"""))

In [13]:
# iter() and next() — the two built-in functions that drive the protocol
nums = [10, 20, 30, 40, 50]

# A list is an iterable but NOT an iterator
print("list has __iter__ :", hasattr(nums, '__iter__'))
print("list has __next__ :", hasattr(nums, '__next__'))
print()

# iter() returns an iterator
it = iter(nums)
print("iterator object  :", it)
print("iterator has __iter__ :", hasattr(it, '__iter__'))
print("iterator has __next__ :", hasattr(it, '__next__'))
print()

# next() advances the cursor
print("next(it) :", next(it))   # 10
print("next(it) :", next(it))   # 20
print("next(it) :", next(it))   # 30
print()

# Default value on exhaustion
it2 = iter([1])
print("next with default:", next(it2, 'DONE'))  # 1
print("next with default:", next(it2, 'DONE'))  # DONE  (no StopIteration)
print()

# iter(obj).__iter__() returns the same object
it3 = iter(nums)
print("iter(it) is it   :", iter(it3) is it3)   # True — iterator is its own iterator

list has __iter__ : True
list has __next__ : False

iterator object  : <list_iterator object at 0x73fc6c6306d0>
iterator has __iter__ : True
iterator has __next__ : True

next(it) : 10
next(it) : 20
next(it) : 30

next with default: 1
next with default: DONE

iter(it) is it   : True


## How to Implement — Custom Iterator Class

Any class that defines both `__iter__` and `__next__` satisfies the iterator protocol.

```python
class MyIterator:
    def __iter__(self):
        return self          # iterator is its own iterator

    def __next__(self):
        if <exhausted>:
            raise StopIteration
        return <next_value>  # advance internal state, return value
```

To make a **reusable iterable** (like a list), implement `__iter__` in a *separate* container class that returns a *fresh iterator object* each time:

In [14]:
# Pattern 1: Combined iterable + iterator (single class)
class CountUp:
    """Counts from start to stop (inclusive)."""
    def __init__(self, start, stop):
        self.current = start
        self.stop    = stop

    def __iter__(self):
        return self          # iterator IS the object itself

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


for n in CountUp(3, 7):
    print(n, end='  ')
print()

# Limitation: single-pass — second loop gives nothing
counter = CountUp(1, 4)
print('First  pass:', list(counter))   # [1, 2, 3, 4]
print('Second pass:', list(counter))   # []  — cursor is already past the end

3  4  5  6  7  
First  pass: [1, 2, 3, 4]
Second pass: []


In [15]:
# Pattern 2: Separate iterable + iterator classes (reusable)
class CountRange:
    """Reusable iterable: each iter() call gets a fresh iterator."""
    def __init__(self, start, stop):
        self.start = start
        self.stop  = stop

    def __iter__(self):
        return _CountRangeIter(self.start, self.stop)  # fresh each time


class _CountRangeIter:
    def __init__(self, start, stop):
        self.current = start
        self.stop    = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


rng = CountRange(1, 5)
print('First  pass:', list(rng))   # [1, 2, 3, 4, 5]
print('Second pass:', list(rng))   # [1, 2, 3, 4, 5]  — fresh iterator every time
print('Third  pass:', list(rng))   # [1, 2, 3, 4, 5]

First  pass: [1, 2, 3, 4, 5]
Second pass: [1, 2, 3, 4, 5]
Third  pass: [1, 2, 3, 4, 5]


## Infinite Iterators

An iterator has no obligation to ever raise `StopIteration`. Infinite iterators are perfectly valid — the caller is responsible for stopping (via `break`, `islice`, `take`, etc.).

In [16]:
import itertools

class InfiniteCounter:
    """Counts up from start forever."""
    def __init__(self, start=0, step=1):
        self.n    = start
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        value  = self.n
        self.n += self.step
        return value      # never raises StopIteration


# Safe usage: always pair with a limit
counter = InfiniteCounter(start=0, step=3)
first_10 = list(itertools.islice(counter, 10))
print('First 10 multiples of 3:', first_10)

# Or break manually
counter2 = InfiniteCounter(start=2, step=2)
evens_under_20 = []
for n in counter2:
    if n >= 20:
        break
    evens_under_20.append(n)
print('Even numbers under 20  :', evens_under_20)

First 10 multiples of 3: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]
Even numbers under 20  : [2, 4, 6, 8, 10, 12, 14, 16, 18]


## Built-in Iterables and Iterators

Python's standard library is built around the iterator protocol.

| Object | Iterable? | Iterator? | Notes |
|---|---|---|---|
| `list`, `tuple`, `set` | ✓ | ✗ | `iter()` returns a fresh `list_iterator` |
| `dict` | ✓ | ✗ | iterates over keys by default |
| `str` | ✓ | ✗ | iterates character by character |
| `range` | ✓ | ✗ | creates `range_iterator` on demand |
| `enumerate` | ✓ | ✓ | lazy index+value pairs |
| `zip` | ✓ | ✓ | stops at shortest input |
| `map`, `filter` | ✓ | ✓ | lazy function application |
| `reversed` | ✓ | ✓ | requires `__reversed__` or `__len__`+`__getitem__` |
| file object | ✓ | ✓ | yields one line per `next()` call |
| generator | ✓ | ✓ | produced by `yield` or generator expression |

In [17]:
# enumerate — adds index
fruits = ['apple', 'banana', 'cherry']
for i, fruit in enumerate(fruits, start=1):
    print(f'  {i}. {fruit}')
print()

# zip — pairs up iterables (stops at shortest)
names  = ['Alice', 'Bob', 'Carol']
scores = [92, 85, 78, 99]          # one extra entry — ignored
for name, score in zip(names, scores):
    print(f'  {name}: {score}')
print()

# map — lazy function application
squares = map(lambda x: x**2, range(1, 6))
print('map object      :', squares)
print('materialised    :', list(squares))
print()

# filter — lazy predicate
evens = filter(lambda x: x % 2 == 0, range(10))
print('filter evens    :', list(evens))
print()

# reversed — backward traversal
print('reversed list   :', list(reversed([1, 2, 3, 4, 5])))
print()

# dict iteration — keys, values, items
d = {'x': 10, 'y': 20, 'z': 30}
print('dict keys  :', list(d))              # same as d.keys()
print('dict values:', list(d.values()))
print('dict items :', list(d.items()))

  1. apple
  2. banana
  3. cherry

  Alice: 92
  Bob: 85
  Carol: 78

map object      : <map object at 0x73fc6c630f70>
materialised    : [1, 4, 9, 16, 25]

filter evens    : [0, 2, 4, 6, 8]

reversed list   : [5, 4, 3, 2, 1]

dict keys  : ['x', 'y', 'z']
dict values: [10, 20, 30]
dict items : [('x', 10), ('y', 20), ('z', 30)]


## `itertools` — Building Blocks for Iterators

`itertools` provides fast, memory-efficient tools for working with iterators. All functions return lazy iterators.

| Category | Functions |
|---|---|
| **Infinite** | `count(start, step)`, `cycle(it)`, `repeat(val, n)` |
| **Finite** | `chain(*its)`, `islice(it, n)`, `takewhile(pred, it)`, `dropwhile(pred, it)`, `compress(it, selectors)` |
| **Combinatoric** | `product(*its)`, `permutations(it, r)`, `combinations(it, r)` |

In [18]:
import itertools

# chain — flatten multiple iterables
chained = list(itertools.chain([1, 2], [3, 4], [5]))
print('chain         :', chained)

# islice — lazy slice (works on infinite iterators)
naturals = itertools.count(1)
print('first 8 primes (islice):', list(itertools.islice(naturals, 8)))

# takewhile — stop as soon as predicate fails
data = [2, 4, 6, 7, 8, 10]
print('takewhile even:', list(itertools.takewhile(lambda x: x % 2 == 0, data)))

# dropwhile — skip while predicate holds, then yield everything
print('dropwhile even:', list(itertools.dropwhile(lambda x: x % 2 == 0, data)))

# cycle — repeat a finite iterable forever
colors = list(itertools.islice(itertools.cycle(['R', 'G', 'B']), 9))
print('cycle R,G,B ×3:', colors)

# product — cartesian product
print('product [1,2]×[a,b]:', list(itertools.product([1, 2], ['a', 'b'])))

# combinations / permutations
print('combinations(3,2)  :', list(itertools.combinations([1, 2, 3], 2)))
print('permutations(3,2)  :', list(itertools.permutations([1, 2, 3], 2)))

chain         : [1, 2, 3, 4, 5]
first 8 primes (islice): [1, 2, 3, 4, 5, 6, 7, 8]
takewhile even: [2, 4, 6]
dropwhile even: [7, 8, 10]
cycle R,G,B ×3: ['R', 'G', 'B', 'R', 'G', 'B', 'R', 'G', 'B']
product [1,2]×[a,b]: [(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
combinations(3,2)  : [(1, 2), (1, 3), (2, 3)]
permutations(3,2)  : [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]


In [19]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 200" width="740" height="200"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="it1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
  </defs>
  <rect width="740" height="200" fill="#f9f9fb" rx="10"/>
  <text x="370" y="26" text-anchor="middle" font-size="14" font-weight="bold" fill="#1a1a2e">itertools Pipeline &#8212; All Lazy, All O(1) Memory</text>

  <rect x="16"  y="50" width="110" height="60" rx="7" fill="#e8ecf5" stroke="#555" stroke-width="1.6"/>
  <text x="71"  y="77" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">count(1)</text>
  <text x="71"  y="93" text-anchor="middle" font-size="9"  fill="#555">1, 2, 3, &#8230;</text>

  <line x1="128" y1="80" x2="168" y2="80" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#it1)"/>

  <rect x="170" y="50" width="130" height="60" rx="7" fill="#dce9f7" stroke="#2166ac" stroke-width="1.6"/>
  <text x="235" y="77" text-anchor="middle" font-size="11" font-weight="bold" fill="#2166ac">islice(it, 20)</text>
  <text x="235" y="93" text-anchor="middle" font-size="9"  fill="#2166ac">first 20 values</text>

  <line x1="302" y1="80" x2="342" y2="80" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#it1)"/>

  <rect x="344" y="50" width="150" height="60" rx="7" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.6"/>
  <text x="419" y="77" text-anchor="middle" font-size="11" font-weight="bold" fill="#2e7d32">filter(is_odd)</text>
  <text x="419" y="93" text-anchor="middle" font-size="9"  fill="#2e7d32">1, 3, 5, 7, 9 &#8230;</text>

  <line x1="496" y1="80" x2="536" y2="80" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#it1)"/>

  <rect x="538" y="50" width="130" height="60" rx="7" fill="#fff3cd" stroke="#e6a817" stroke-width="1.6"/>
  <text x="603" y="77" text-anchor="middle" font-size="11" font-weight="bold" fill="#856404">map(x&#178;)</text>
  <text x="603" y="93" text-anchor="middle" font-size="9"  fill="#856404">1, 9, 25, 49 &#8230;</text>

  <line x1="670" y1="80" x2="710" y2="80" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#it1)"/>
  <text x="725" y="84" text-anchor="middle" font-size="9" fill="#3a8c3f">list()</text>

  <text x="370" y="140" text-anchor="middle" font-size="10" fill="#555">Each stage is a lazy iterator. No intermediate list is created. Memory cost = O(1) across the entire pipeline.</text>
  <rect x="16" y="152" width="708" height="36" rx="6" fill="#e8ecf5" stroke="#888" stroke-width="1"/>
  <text x="370" y="168" text-anchor="middle" font-size="10" fill="#333">list(map(lambda x: x**2, filter(lambda x: x%2, itertools.islice(itertools.count(1), 20))))</text>
  <text x="370" y="182" text-anchor="middle" font-size="9" fill="#555">&#8594; [1, 9, 25, 49, 81, 121, 169, 225, 289, 361]</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  Values flow right one at a time; no stage buffers the stream.
</figcaption>
</figure>
"""))

## Advantages

| # | Advantage | Detail |
|---|---|---|
| 1 | **Uniform traversal interface** | Any object that implements the protocol works with `for`, `list()`, `sum()`, `zip()`, etc. |
| 2 | **Memory efficiency** | One value in memory at a time — identical to generators |
| 3 | **Infinite sequences** | No upper bound required; caller decides when to stop |
| 4 | **Reusability** | Separate iterable + iterator classes allow multiple independent traversals |
| 5 | **Composability** | Iterators chain with `itertools`, `map`, `filter`, `zip` naturally |
| 6 | **Extra methods** | A class can expose `__len__`, `__contains__`, `peek()`, `reset()` alongside the protocol |
| 7 | **Lazy evaluation** | Work is deferred to consumption; expensive computation is skipped if not needed |

## Disadvantages

| # | Disadvantage | Detail |
|---|---|---|
| 1 | **Single-pass (iterators)** | Once `StopIteration` is raised, the iterator is exhausted — `next()` keeps raising it |
| 2 | **No random access** | `it[3]` raises `TypeError` |
| 3 | **No `len()`** | `len(iterator)` raises `TypeError`; size is unknown until fully consumed |
| 4 | **Class boilerplate** | A custom iterator class requires `__iter__` + `__next__` + state management; a generator does the same in 2 lines |
| 5 | **No look-ahead** | Cannot inspect the next value without consuming it (use `itertools.tee` or a wrapper) |
| 6 | **Debugging difficulty** | Errors surface at consumption time, making the stack trace point far from the original source |

In [20]:
# Disadvantage 1 — exhausted iterators stay exhausted
it = iter([1, 2, 3])
print('consumed:', list(it))     # [1, 2, 3]
print('again   :', list(it))     # []  — exhausted
print()

# Disadvantage 2 & 3 — no indexing, no len
it2 = iter([10, 20, 30])
try:    print(it2[0])
except TypeError as e: print(f'Index  error: {e}')
try:    print(len(it2))
except TypeError as e: print(f'len    error: {e}')
print()

# Disadvantage 5 — no peek without consuming
import itertools
it3 = iter([5, 10, 15])
it3a, it3b = itertools.tee(it3)        # creates two independent copies (stores values)
peek = next(it3a)
print(f'Peeked: {peek}  — full stream: {list(it3b)}')
print()

# Fix for disadvantage 4 — use a generator instead of a class
def count_up_gen(start, stop):
    while start <= stop:
        yield start
        start += 1

print('Generator equivalent:', list(count_up_gen(3, 7)))

consumed: [1, 2, 3]
again   : []

Index  error: 'list_iterator' object is not subscriptable
len    error: object of type 'list_iterator' has no len()

Peeked: 5  — full stream: [5, 10, 15]

Generator equivalent: [3, 4, 5, 6, 7]


## Iterator vs Generator

Generators automatically implement the iterator protocol — every generator is an iterator. The choice is about **how much control** you need.

Use a **generator** when:
- The logic fits naturally with `yield`
- You don't need to reset or share state
- You want minimal code

Use a **custom iterator class** when:
- You need to **reset** the iterator (`reset()` method)
- You want to expose **extra methods** (`peek()`, `__len__`, `__contains__`)
- State is complex enough to benefit from a proper object model

In [21]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 310" width="740" height="310"
     font-family="Segoe UI, Arial, sans-serif">
  <rect width="740" height="310" fill="#f9f9fb" rx="10"/>
  <text x="370" y="28" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Iterator Class vs Generator</text>

  <!-- Header row -->
  <rect x="16"  y="44" width="230" height="32" rx="5" fill="#555"/>
  <text x="131" y="65" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Feature</text>
  <rect x="252" y="44" width="225" height="32" rx="5" fill="#c0392b"/>
  <text x="365" y="65" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Custom Iterator Class</text>
  <rect x="483" y="44" width="241" height="32" rx="5" fill="#2e7d32"/>
  <text x="604" y="65" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Generator</text>

  <!-- Row data -->
  <!-- row 1 -->
  <rect x="16" y="80" width="708" height="30" rx="0" fill="#f0f0f0"/>
  <text x="131" y="99" text-anchor="middle" font-size="10" fill="#333">Syntax</text>
  <text x="365" y="99" text-anchor="middle" font-size="10" fill="#333">class + __iter__ + __next__</text>
  <text x="604" y="99" text-anchor="middle" font-size="10" fill="#333">def + yield</text>
  <!-- row 2 -->
  <rect x="16" y="110" width="708" height="30" rx="0" fill="#fafafa"/>
  <text x="131" y="129" text-anchor="middle" font-size="10" fill="#333">Code volume</text>
  <text x="365" y="129" text-anchor="middle" font-size="10" fill="#c0392b">High (10&#8211;20 lines)</text>
  <text x="604" y="129" text-anchor="middle" font-size="10" fill="#2e7d32">Low (2&#8211;5 lines)</text>
  <!-- row 3 -->
  <rect x="16" y="140" width="708" height="30" rx="0" fill="#f0f0f0"/>
  <text x="131" y="159" text-anchor="middle" font-size="10" fill="#333">State management</text>
  <text x="365" y="159" text-anchor="middle" font-size="10" fill="#333">Manual (self.x)</text>
  <text x="604" y="159" text-anchor="middle" font-size="10" fill="#2e7d32">Automatic (stack frame)</text>
  <!-- row 4 -->
  <rect x="16" y="170" width="708" height="30" rx="0" fill="#fafafa"/>
  <text x="131" y="189" text-anchor="middle" font-size="10" fill="#333">Memory</text>
  <text x="365" y="189" text-anchor="middle" font-size="10" fill="#333">O(1)</text>
  <text x="604" y="189" text-anchor="middle" font-size="10" fill="#333">O(1)</text>
  <!-- row 5 -->
  <rect x="16" y="200" width="708" height="30" rx="0" fill="#f0f0f0"/>
  <text x="131" y="219" text-anchor="middle" font-size="10" fill="#333">Reusable iterable</text>
  <text x="365" y="219" text-anchor="middle" font-size="10" fill="#2e7d32">Yes (separate class pattern)</text>
  <text x="604" y="219" text-anchor="middle" font-size="10" fill="#c0392b">No (re-call function)</text>
  <!-- row 6 -->
  <rect x="16" y="230" width="708" height="30" rx="0" fill="#fafafa"/>
  <text x="131" y="249" text-anchor="middle" font-size="10" fill="#333">Extra methods</text>
  <text x="365" y="249" text-anchor="middle" font-size="10" fill="#2e7d32">Yes (peek, reset, __len__)</text>
  <text x="604" y="249" text-anchor="middle" font-size="10" fill="#c0392b">No</text>
  <!-- row 7 -->
  <rect x="16" y="260" width="708" height="30" rx="0" fill="#f0f0f0"/>
  <text x="131" y="279" text-anchor="middle" font-size="10" fill="#333">send() / two-way comms</text>
  <text x="365" y="279" text-anchor="middle" font-size="10" fill="#c0392b">No (not built-in)</text>
  <text x="604" y="279" text-anchor="middle" font-size="10" fill="#2e7d32">Yes (yield as expression)</text>

  <!-- Grid lines -->
  <line x1="246" y1="44" x2="246" y2="290" stroke="#ccc" stroke-width="1"/>
  <line x1="477" y1="44" x2="477" y2="290" stroke="#ccc" stroke-width="1"/>
  <rect x="16" y="44" width="708" height="246" rx="0" fill="none" stroke="#ccc" stroke-width="1"/>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  Generators are the default choice; reach for an iterator class only when you need reusability or extra methods.
</figcaption>
</figure>
"""))

In [22]:
# Same task — iterator class vs generator

# ── Iterator class version ──────────────────────────────────────────────
class EvenNumbers:
    def __init__(self, limit):
        self.current = 0
        self.limit   = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration
        value = self.current
        self.current += 2
        return value

    def reset(self):              # bonus: class allows reset
        self.current = 0


# ── Generator version ───────────────────────────────────────────────────
def even_numbers(limit):
    n = 0
    while n <= limit:
        yield n
        n += 2


# Both produce identical output
print('Class    :', list(EvenNumbers(10)))
print('Generator:', list(even_numbers(10)))
print()

# Class bonus: reset()
ev = EvenNumbers(8)
print('Pass 1:', list(ev))
ev.reset()
print('Pass 2:', list(ev))   # works after reset

Class    : [0, 2, 4, 6, 8, 10]
Generator: [0, 2, 4, 6, 8, 10]

Pass 1: [0, 2, 4, 6, 8]
Pass 2: [0, 2, 4, 6, 8]


## Summary

| | Iterable | Iterator |
|---|---|---|
| Implements | `__iter__()` | `__iter__()` + `__next__()` |
| `__iter__()` returns | fresh iterator | `self` |
| Reusable | Yes | No (unless reset manually) |
| `next()` works | No | Yes |
| Examples | `list`, `dict`, `str`, `range` | `enumerate`, `zip`, `map`, files, generators |

**Decision guide:**

```
Need to iterate lazily?
├── Simple linear logic → use a generator (yield)
└── Need reset / extra methods / reusable container?
    ├── Yes → custom iterator class (separate iterable + iterator)
    └── No  → generator is fine
```

Both generators and iterator classes satisfy the same protocol and work identically with `for`, `list()`, `sum()`, `zip()`, and all of `itertools`.